In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Cargar dataset de alimentadores hospitalarios
csv_path = os.path.join("..", "data", "schedules", "cable_schedule.csv")
df = pd.read_csv(csv_path)

print(f"Dataset cargado exitosamente: {df.shape[0]} circuitos de alimentadores.")
df.head()

In [ ]:
print("=== RESUMEN TÉCNICO DE CONDUCTORES ===")
print(df.info())
print("\n=== DISTRIBUCIÓN POR SISTEMA HOSPITALARIO ===")
print(df['sistema'].value_counts())
print("\n=== ESTADÍSTICAS DE DIÁMETROS Y RADIOS MÍNIMOS ===")
display(df[['diametro_ext_mm', 'peso_lineal_kg_m', 'seccion_mm2', 'radio_min_curv_mm']].describe())

In [ ]:
# 1. Feature de Área Total de Cobre por Alimentador (mm2)
df['area_cobre_total_mm2'] = df['num_conductores'] * (np.pi * (df['diametro_ext_mm'] / 2)**2)

# 2. Estimación de Ancho Mínimo de Bandeja (NEC Art. 392 - Llenado máx 40% o monocapa)
# Bandeja estándar con peralte útil de 100 mm: Area_util = Ancho * 100 * 0.40
ancho_bandeja_ref_mm = 300.0  # Referencia: bandeja de 300x100 mm
area_util_ref = ancho_bandeja_ref_mm * 100.0 * 0.40
df['R_fill_ref'] = df['area_cobre_total_mm2'] / area_util_ref

# 3. Margen de Curvatura Admisible (MS_curv) para un accesorio estándar de R = 300 mm
R_accesorio_ref_mm = 300.0
df['MS_curv_ref'] = 1.0 - (df['radio_min_curv_mm'] / R_accesorio_ref_mm)

print("Features de dominio calculadas:")
df[['circuito_id', 'calibre_fase', 'area_cobre_total_mm2', 'R_fill_ref', 'MS_curv_ref']].head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Split por grupos / estratificado por sistema eléctrico
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['sistema'])

# Escalado Min-Max ajustado ÚNICAMENTE con el conjunto de entrenamiento (CERO LEAKAGE)
scaler = MinMaxScaler()
scaler.fit(train_df[['diametro_ext_mm', 'peso_lineal_kg_m']])

# Transformación posterior sobre train y test
train_scaled = scaler.transform(train_df[['diametro_ext_mm', 'peso_lineal_kg_m']])
test_scaled = scaler.transform(test_df[['diametro_ext_mm', 'peso_lineal_kg_m']])

print("Protocolo Cero Data Leakage verificado:")
print(f"Muestras de Train: {train_df.shape[0]} | Muestras de Test: {test_df.shape[0]}")
print(f"Escalador ajustado estrictamente sobre Train (Valores Min-Max de Train):")
print(f"Min: {scaler.data_min_} | Max: {scaler.data_max_}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico A: Distribución de calibres y radios de curvatura mínimos
sns.barplot(data=df, x='calibre_fase', y='radio_min_curv_mm', hue='sistema', ax=axes[0], ci=None)
axes[0].axhline(300, color='red', linestyle='--', label='Codo comercial R=300mm')
axes[0].set_title("Radio Mínimo de Flexión por Calibre (N2XOH)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Radio Mínimo Adm. (mm)")
axes[0].set_xlabel("Calibre Comercial")
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title="Sistema")

# Gráfico B: Dispersión Diámetro vs. Peso Lineal
sns.scatterplot(data=df, x='diametro_ext_mm', y='peso_lineal_kg_m', hue='sistema', style='calibre_fase', s=100, ax=axes[1])
axes[1].set_title("Relación Diámetro Exterior vs. Carga Lineal", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Diámetro Exterior (mm)")
axes[1].set_ylabel("Peso Lineal (kg/m)")

plt.tight_layout()
plt.show()